In [1]:
#imports
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import plotly.figure_factory as ff
import plotly.io as pio

In [2]:
# Paths
pth_emb_npy = "/group/glastonbury/sara/data/latent_factors/time2ch_4ChTrans_FactorVAE_latent64.npy"
outpth = "/project/ukbblatent/processed_latents"

pth_tstSubs_cycles = '/group/glastonbury/sara/data/subjects/test_subIDs_Acqs_nCardiacCycles_4Chtransverse_0.json'
pth_tst_subIDs = '/group/glastonbury/sara/data/subjects/test_subIDs_201x208.json'

pth_baseline = '/project/ukbblatent/clinicaldata/baseline.tsv'
pth_dob = "/project/ukbblatent/clinicaldata/dateofbirth.tsv"
pth_cardiac_measures = "/project/ukbblatent/clinicaldata/cardiac_measures.tsv"
pth_cardiacKV = '/group/glastonbury/ukbb-clinical-data/derived-cardiac-measures-key-values.parquet'
pth_body_impedance = "/project/ukbblatent/clinicaldata/body_impedance.tsv" 
pth_body_measure = "/project/ukbblatent/clinicaldata/body_measure.tsv"

In [3]:
# fetch the test IDs
testIDs = json.load(open(pth_tst_subIDs, 'r'))

In [4]:
# Read and prepare the latent embeddings
processed_emb = np.load(pth_emb_npy, allow_pickle=True)
processed_emb = pd.DataFrame(processed_emb, columns=['id', 'dataset', 'MRIvisit', 'data_tag', 'Zs'])
processed_emb = processed_emb[processed_emb.id.isin(list(testIDs))]
processed_emb = processed_emb.set_index('id')
processed_emb.index = processed_emb.index.astype(int)

processed_emb.Zs = processed_emb.Zs.apply(np.squeeze)
Zs_df = processed_emb.Zs.apply(pd.Series)
Zs_df.columns = [f'Z{i}' for i in range(len(Zs_df.columns))]
processed_emb = pd.concat([processed_emb.drop('Zs', axis=1), Zs_df], axis=1)

processed_emb.MRIvisit = processed_emb.MRIvisit.apply(lambda x: str(x)[0]).astype('int')

In [5]:
# read and add cardiac cycles to the dataframe of embeddings
cycles = json.load(open(pth_tstSubs_cycles, 'r'))
cycles_dict = {int(v): int(k) for k, lst in cycles.items() for v in lst}
processed_emb['cardiac_cycles'] = processed_emb.index.map(lambda x: cycles_dict.get(x))

In [128]:
#read and process the baselines
baseline = pd.read_table(pth_baseline).set_index('f.eid')
dob = pd.read_table(pth_dob).set_index('f.eid')
baseline = dob.join(baseline)

# filter based on the required IDs
baseline = baseline[baseline.index.isin(processed_emb.index)]

# BASELINE IDs
# 34:    year of birth
# 52:    month of birth
# 31:    sex (0 = female, 1 = male)
# 20116: smoking status
# 21000: ethnic background
# 21001: BMI
# 21022: age at the recruitment
# 40000: date of death
# 40001: primary cause of death


# drop useless columns
baseline = baseline.drop(['f.21000.1.0', 'f.21000.2.0', 'f.20116.3.0', 'f.21001.3.0', 'f.40000.1.0', 'f.40001.1.0'], axis=1)
# 21000.1, 21000.2, 21000.3 - ethnicity second assessment visit and imaging visits (ethnic background does not change) 
# 20116.3, 21001.3 - Smoking and BMI at second imaging visit (3) are NaN
# 40000.1, 40001.1 - second column of death date and second column of primary death cause (first col is enough)


# change column names 
baseline_cols = {'f.34.0.0':    'Birth_Month',
                 'f.52.0.0':    'Birth_Year',
                 'f.31.0.0':    'Gender',
                 'f.20116.0.0': 'Smoking_0', # first assessment visit
                 'f.20116.1.0': 'Smoking_1', # second assessment visit
                 'f.20116.2.0': 'Smoking_Imaging', # MRI imaging (first visit)
                 'f.21000.0.0': 'Ethnicity', 
                 'f.21001.0.0': 'BMI_0', 
                 'f.21001.1.0': 'BMI_1', 
                 'f.21001.2.0': 'BMI_Imaging', 
                 'f.21022.0.0': 'Age_Recruitment',
                 'f.40000.0.0': 'Death_Date',
                 'f.40001.0.0': 'Death_Cause',
                }

baseline = baseline.rename(columns=baseline_cols)


# fill NaN values
baseline['Smoking_1'] = baseline['Smoking_1'].fillna(baseline['Smoking_0']) # if smoking has not been updated - keep previous data
baseline['Smoking_Imaging'] = baseline['Smoking_Imaging'].fillna(baseline['Smoking_1'])

baseline['BMI_1'] = baseline['BMI_1'].fillna(baseline['BMI_0'])  # if BMI measure not updated - keep previous measure
baseline['BMI_Imaging']= baseline['BMI_Imaging'].fillna(baseline['BMI_1']) #fill NaN values in BMI at imaging with first BMI

# drop columns not needed anymore
baseline = baseline.drop(['Smoking_0', 'Smoking_1', 'BMI_0', 'BMI_1'], axis=1)

# add categories for BMI
baseline['BMICAT_Imaging'] = pd.cut(baseline['BMI_Imaging'], bins=[0,18.5,24.9,29.9,60], labels=['Underweight', 'Healthy', 'Overweight', 'Obese'])

# add categories for Ethnicity
ethnic_dict = {'1':'White', '2':'Mixed', '3':'S.Asian', '4':'Black', '5':'E.Asian', '6':'Other', '-':'Other', 'n':'Other'}
baseline['Ethnicity'] = baseline.Ethnicity.apply(lambda x: ethnic_dict.get(str(x)[0]))

baseline_alive = baseline[baseline['Death_Date'].isna()].drop(['Death_Date', 'Death_Cause'], axis=1)
baseline_dead = baseline[baseline['Death_Date'].notna()]
baseline = baseline.drop(['Death_Date', 'Death_Cause'], axis=1) # just going to ignore the death columns

In [129]:
# read and process the clinical data: cardiac measures
cardiac_measures = pd.read_table(pth_cardiac_measures).set_index('f.eid')

# filter based on the required IDs
cardiac_measures = cardiac_measures[cardiac_measures.index.isin(processed_emb.index)]

# drop NaN or useless columns (Sara: please put the column names as a comment)
cardiac_measures = cardiac_measures.drop(['f.12687.2.1', 'f.12687.2.2', 'f.12687.2.3', 'f.12687.2.4', 'f.12687.3.1', 'f.12687.3.2', 'f.12687.3.3', 'f.12687.3.4'], axis=1)
# 12687: mean arterial pressure - keep only the first measurement for each MRI visit, drop the other columns (mostly NaN)


cmeas_keys = pd.read_parquet(pth_cardiacKV, engine='fastparquet')

# keys dictionary
keys = {cmeas_keys.loc[idx]['field_id']: cmeas_keys.loc[idx]['field_name'] for idx in cmeas_keys.index}
keys['f.12687.2.0'] = 'Mean arterial pressure'
keys['f.12687.3.0'] = 'Mean arterial pressure'
keys['f.22427.2.0'] = 'Body surface area'
keys['f.22427.3.0'] = 'Body surface area'

cardiac_measures = pd.concat([cardiac_measures.filter(regex='2\.0$').dropna().rename(columns=keys),
                              cardiac_measures.filter(regex='3\.0$').dropna().rename(columns=keys)])

# Add categorical variables for Ejection Fraction
cardiac_measures['LVEFCAT'] = pd.cut(cardiac_measures['LV ejection fraction'], bins=[20,40,50,80], labels=['Sev. Reduced', 'Reduced', 'Normal'])


In [130]:
# read and process the clinical data: other available ones (TODO)
body_impedance = pd.read_table(pth_body_impedance).set_index('f.eid') 
body_measure = pd.read_table(pth_body_measure).set_index('f.eid')

In [131]:
processed_emb = processed_emb.drop(columns=['dataset', 'data_tag']) #we don't need them for now, for these analyses at least

# drop any NaNs to simplify the analysis
baseline = baseline.dropna()
baseline_alive = baseline_alive.dropna()
baseline_dead = baseline_dead.dropna()
cardiac_measures = cardiac_measures.dropna()

# join the dataframes - emb and baseline
baseline_emb = baseline.join(processed_emb).dropna()
baseline_alive_emb = baseline_alive.join(processed_emb).dropna()
baseline_dead_emb = baseline_dead.join(processed_emb).dropna()

# join the dataframes - emb and cardiac measures
cardiac_emb = cardiac_measures.join(processed_emb).dropna()

# join the dataframes - emb, baseline and cardiac measures
cardiac_baseline_emb = cardiac_measures.join(baseline_emb).dropna()
cardiac_baseline_alive_emb = cardiac_measures.join(baseline_alive_emb).dropna()
cardiac_baseline_dead_emb = cardiac_measures.join(baseline_dead_emb).dropna()

In [132]:
# Save the created dataframes
outpth_df = f"{outpth}/{os.path.basename(pth_emb_npy).split('.')[0]}/dataframes"
os.makedirs(outpth_df, exist_ok=True)
baseline_emb.to_csv(f'{outpth_df}/baseline_emb.csv')
baseline_alive_emb.to_csv(f'{outpth_df}/baseline_alive_emb.csv')
baseline_dead_emb.to_csv(f'{outpth_df}/baseline_dead_emb.csv')
cardiac_emb.to_csv(f'{outpth_df}/cardiac_emb.csv')
cardiac_baseline_emb.to_csv(f'{outpth_df}/cardiac_baseline_emb.csv')
cardiac_baseline_alive_emb.to_csv(f'{outpth_df}/cardiac_baseline_alive_emb.csv')
cardiac_baseline_dead_emb.to_csv(f'{outpth_df}/cardiac_baseline_dead_emb.csv')

In [133]:
def PlotNSaveHeatmap(matrix, title, fig_size=1500, font_size=8, rootpath=""):
    matrix_rounded = matrix.round(2)

    fig = ff.create_annotated_heatmap(
        z=matrix_rounded.values,
        x=matrix_rounded.columns.tolist(),
        y=matrix_rounded.index.tolist(),
        annotation_text=matrix_rounded.values,
        colorscale='Viridis',
        hoverinfo='z',
        showscale=True
    )

    for i in range(len(fig.layout.annotations)):
        fig.layout.annotations[i].font.size = font_size

    fig.update_layout(
        title=title,
        width=fig_size,
        height=fig_size,
        xaxis=dict(side='bottom'),
        margin=dict(t=100)
    )

    pio.write_html(fig, file=f'{rootpath}{title.replace(" ", "_")}.html', auto_open=False)

In [135]:
#Co-relation analysis
outpth_correl = f"{outpth}/{os.path.basename(pth_emb_npy).split('.')[0]}/correlation_analysis/"
os.makedirs(outpth_correl, exist_ok=True)

# Convert the categorial variables to one-hot encoding
cardiac_baseline_emb_1HOT = pd.get_dummies(cardiac_baseline_emb, columns=[col for col in cardiac_baseline_emb.columns if 'CAT' in col])

correlation_matrix = cardiac_baseline_emb_1HOT.corr()
spearman_corr = cardiac_baseline_emb_1HOT.corr(method='spearman')
kendall_corr = cardiac_baseline_emb_1HOT.corr(method='kendall')

PlotNSaveHeatmap(correlation_matrix, 'CardiacBaseline Correlation Matrix Heatmap', fig_size=1500, font_size=8, rootpath=outpth_correl)
PlotNSaveHeatmap(spearman_corr, 'CardiacBaseline Spearman Rank Correlation Matrix Heatmap', fig_size=1500, font_size=8, rootpath=outpth_correl)
PlotNSaveHeatmap(kendall_corr, 'CardiacBaseline Kendall Tau Rank Correlation Matrix Heatmap', fig_size=1500, font_size=8, rootpath=outpth_correl)

In [18]:
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

data = pd.read_csv(r"C:\MyFiles\UKBB_processed_latents\time2ch_4ChTrans_FactorVAE_latent64\dataframes\cardiac_baseline_emb.csv", index_col=0)

attributes2ignore = ["Birth_Month", "Ethnicity", "MRIvisit"]
binary_categorical_data = ["Gender", "Smoking_Imaging"] 

latent_factors = [f"Z{i}" for i in range(64)]
multi_class_categorical_data = [c for c in data.columns if "CAT" in c]
continuous_clinical_data = [c for c in data.columns if c not in attributes2ignore+latent_factors+multi_class_categorical_data+binary_categorical_data]


In [67]:
def calculate_correlations(df, factors, continuous_vars, binary_vars, multi_class_vars):
    correlations = {}

    # Continuous clinical data
    for factor in factors:
        for var in continuous_vars:
            pearson_corr, pearson_pval = stats.pearsonr(df[factor], df[var])
            spearman_corr, spearman_pval = stats.spearmanr(df[factor], df[var])
            correlations[(factor, var, 'pearson')] = (pearson_corr, pearson_pval)
            correlations[(factor, var, 'spearman')] = (spearman_corr, spearman_pval)

    # Binary categorical clinical data
    for factor in factors:
        for var in binary_vars:
            point_biserial_corr, point_biserial_pval = stats.pointbiserialr(df[factor], df[var])
            correlations[(factor, var, 'point_biserial')] = (point_biserial_corr, point_biserial_pval)

    # Multi-class categorical clinical data
    for factor in factors:
        for var in multi_class_vars:
            model = ols(f"{factor} ~ C({var})", data=df).fit()
            anova_table = anova_lm(model)
            eta_squared = anova_table['sum_sq'][0] / (anova_table['sum_sq'][0] + anova_table['sum_sq'][1])
            eta_squared_pval = anova_table['PR(>F)'][0]
            correlations[(factor, var, 'eta_squared')] = (eta_squared, eta_squared_pval)

    return correlations

def correlations_to_dataframe(correlations):
    rows = []
    for key, value in correlations.items():
        corr_coeff, p_val = value
        rows.append([key[0], key[1], key[2], corr_coeff, p_val])

    return pd.DataFrame(rows, columns=['Factor', 'Attribute', 'Correlation_Type', 'Correlation_Coefficient', 'P_Value'])

def categorize_corrcoeef_strength(corr_coeff):
    abs_corr_coeff = abs(corr_coeff)
    if 0.00 <= abs_corr_coeff < 0.20:
        return 'Very weak'
    elif 0.20 <= abs_corr_coeff < 0.40:
        return 'Weak'
    elif 0.40 <= abs_corr_coeff < 0.60:
        return 'Moderate'
    elif 0.60 <= abs_corr_coeff < 0.80:
        return 'Strong'
    else:  # 0.80 <= abs_corr_coeff <= 1.00
        return 'Very strong'

def categories_correlations(df, alpha=0.05):
    df['isSignificant_Correlation'] = df['P_Value'] < alpha
    df['Correlation_Strength'] = df['Correlation_Coefficient'].apply(categorize_corrcoeef_strength)
    return df

correlations = categories_correlations(correlations_to_dataframe(calculate_correlations(data, latent_factors, continuous_clinical_data, binary_categorical_data, multi_class_categorical_data)))


In [70]:
correlations.Correlation_Strength.value_counts()

Very weak    1683
Weak          193
Moderate       41
Strong          3
Name: Correlation_Strength, dtype: int64

In [71]:
correlations[correlations.isSignificant_Correlation].Correlation_Strength.value_counts()

Very weak    1309
Weak          193
Moderate       41
Strong          3
Name: Correlation_Strength, dtype: int64

In [72]:
import plotly.express as px
import plotly.io as pio

def create_heatmaps(correlations, factors, continuous_vars, binary_vars, multi_class_vars):
    # Function to create a DataFrame for each correlation type
    def create_corr_dataframe(correlations, corr_type, index, columns):
        df = pd.DataFrame(index=index, columns=columns)
        for key, value in correlations.items():
            if key[2] == corr_type:
                df.at[key[0], key[1]] = value
        return df

    # Continuous clinical data
    if continuous_vars:
        pearson_df = create_corr_dataframe(correlations, 'pearson', factors, continuous_vars)
        spearman_df = create_corr_dataframe(correlations, 'spearman', factors, continuous_vars)

        pearson_heatmap = px.imshow(pearson_df, color_continuous_scale='Viridis')
        spearman_heatmap = px.imshow(spearman_df, color_continuous_scale='Viridis')

        pio.write_html(pearson_heatmap, file='pearson_heatmap.html')
        pio.write_html(spearman_heatmap, file='spearman_heatmap.html')

    # Binary categorical clinical data
    if binary_vars:
        point_biserial_df = create_corr_dataframe(correlations, 'point_biserial', factors, binary_vars)
        point_biserial_heatmap = px.imshow(point_biserial_df, color_continuous_scale='Viridis')
        pio.write_html(point_biserial_heatmap, file='point_biserial_heatmap.html')

    # Multi-class categorical clinical data
    if multi_class_vars:
        eta_squared_df = create_corr_dataframe(correlations, 'eta_squared', factors, multi_class_vars)
        eta_squared_heatmap = px.imshow(eta_squared_df, color_continuous_scale='Viridis')
        pio.write_html(eta_squared_heatmap, file='eta_squared_heatmap.html')

# Call the function to create and save heatmaps as HTML files
create_heatmaps(correlations, latent_factors, continuous_clinical_data, binary_categorical_data, multi_class_categorical_data)
